In [2]:
"""
build_ecological_zones.py
═════════════════════════════════════════════════════════════════
Builds the 5-zone ecological FeatureCollection locally from the
3 climatic zone shapefiles, then exports as GeoJSON ready to
upload as a GEE asset.

INPUT FILES (in your Maps/Climate Zone folder):
  Sahelian-Desert.shp
  Soudanian_dissolved.shp
  Guinea_dissolved.shp

OUTPUT:
  ecological_zones_5class.geojson   ← upload this to GEE as asset

PROPERTIES on each feature (match the JS dashboard exactly):
  zone_id        int     1–5
  zone_name      str     e.g. 'Sahelian'
  zone_name_fr   str     French name
  source_zone    str     which input shapefile it came from
  color_hex      str     hex colour used in dashboard legend
  rainfall_mm_yr str     rainfall range label
  area_km2       float   area in km²

HOW TO RUN:
  pip install geopandas shapely
  python build_ecological_zones.py

THEN:
  Upload ecological_zones_5class.geojson to GEE Assets as:
  projects/ee-desmond/assets/ecological_zones_5class
═════════════════════════════════════════════════════════════════
"""

import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────

# Folder containing your 3 shapefiles — update if needed
ZONE_DIR = Path(r"C:\Users\Gebruiker\OneDrive\Spain\Paper 1\Cooling-Degree-Day-in-West-Africa\Maps\Climate Zone")

# Output file — saved in the same folder
OUTPUT_FILE = ZONE_DIR / "ecological_zones_5class.geojson"

# Coordinate reference system — GEE expects WGS84
CRS = "EPSG:4326"

# West Africa bounding box (clips any geometry outside study area)
WA_BOUNDS = box(-20, -5, 25, 25)

# ── ZONE DEFINITIONS ──────────────────────────────────────────────────────
# Each entry defines one ecological zone:
#   source_file  : shapefile stem name (without .shp)
#   lat_min/max  : latitude band to clip from the source polygon
#   All other fields match the JS dashboard ECO_ZONE_DEFS exactly

ZONE_DEFS = [
    {
        "zone_id"       : 1,
        "zone_name"     : "Saharian",
        "zone_name_fr"  : "Zone Saharienne",
        "source_file"   : "Sahelian-Desert",
        "lat_min"       : 18.0,   # upper part of Sahelian shapefile
        "lat_max"       : 30.0,
        "color_hex"     : "#F5DEB3",
        "rainfall_mm_yr": "<25 mm/yr",
    },
    {
        "zone_id"       : 2,
        "zone_name"     : "Sahelian",
        "zone_name_fr"  : "Zone Sahélienne",
        "source_file"   : "Sahelian-Desert",
        "lat_min"       : -5.0,   # lower part of Sahelian shapefile
        "lat_max"       : 18.0,
        "color_hex"     : "#E8A838",
        "rainfall_mm_yr": "200-600 mm/yr",
    },
    {
        "zone_id"       : 3,
        "zone_name"     : "Soudanian",
        "zone_name_fr"  : "Zone Soudanienne",
        "source_file"   : "Soudanian_dissolved",
        "lat_min"       : -5.0,   # full extent of Soudanian shapefile
        "lat_max"       : 30.0,
        "color_hex"     : "#CC6600",
        "rainfall_mm_yr": "600-1200 mm/yr",
    },
    {
        "zone_id"       : 4,
        "zone_name"     : "Guinean",
        "zone_name_fr"  : "Zone Guinéenne",
        "source_file"   : "Guinea_dissolved",
        "lat_min"       : 7.0,    # upper part of Guinea shapefile
        "lat_max"       : 30.0,
        "color_hex"     : "#78C850",
        "rainfall_mm_yr": "1200-2000 mm/yr",
    },
    {
        "zone_id"       : 5,
        "zone_name"     : "Guineo-Congolean",
        "zone_name_fr"  : "Zone Guinéo-Congolaise",
        "source_file"   : "Guinea_dissolved",
        "lat_min"       : -5.0,   # lower part of Guinea shapefile
        "lat_max"       : 7.0,
        "color_hex"     : "#1A6B1A",
        "rainfall_mm_yr": ">2000 mm/yr",
    },
]


# ── LOAD AND DISSOLVE EACH SHAPEFILE ─────────────────────────────────────

print("Loading shapefiles …")

shapefiles = {}
for shp_name in ["Sahelian-Desert", "Soudanian_dissolved", "Guinea_dissolved"]:
    shp_path = ZONE_DIR / f"{shp_name}.shp"
    if not shp_path.exists():
        raise FileNotFoundError(
            f"\n❌ Shapefile not found: {shp_path}\n"
            f"   Check that ZONE_DIR is set correctly."
        )
    gdf = gpd.read_file(shp_path).to_crs(CRS)
    # Dissolve all polygons into one — we only need the geometry
    dissolved = gdf.dissolve().geometry.iloc[0]
    # Clip to West Africa bounding box
    dissolved = dissolved.intersection(WA_BOUNDS)
    shapefiles[shp_name] = dissolved
    print(f"  ✓  {shp_name}  ({gdf.shape[0]} feature(s) dissolved)")


# ── BUILD 5 ECOLOGICAL ZONES ─────────────────────────────────────────────

print("\nBuilding ecological zones …")

rows = []
for z in ZONE_DEFS:
    # Source polygon
    source_geom = shapefiles[z["source_file"]]

    # Latitude band polygon
    lat_band = box(-20, z["lat_min"], 25, z["lat_max"])

    # Intersect source with latitude band
    zone_geom = source_geom.intersection(lat_band)

    # Skip if intersection is empty
    if zone_geom.is_empty:
        print(f"  ⚠  {z['zone_name']}: empty geometry — check lat_min/lat_max")
        continue

    # Compute area in km²
    # Use an equal-area projection for accurate area calculation
    area_km2 = (
        gpd.GeoSeries([zone_geom], crs=CRS)
           .to_crs("ESRI:54009")   # Mollweide equal-area
           .area.iloc[0]
        / 1e6
    )

    rows.append({
        "zone_id"       : z["zone_id"],
        "zone_name"     : z["zone_name"],
        "zone_name_fr"  : z["zone_name_fr"],
        "source_zone"   : z["source_file"],
        "color_hex"     : z["color_hex"],
        "rainfall_mm_yr": z["rainfall_mm_yr"],
        "area_km2"      : round(area_km2, 1),
        "geometry"      : zone_geom,
    })
    print(f"  ✓  {z['zone_name']:<20}  area = {area_km2:,.0f} km²")


# ── ASSEMBLE GEODATAFRAME ─────────────────────────────────────────────────

eco_zones = gpd.GeoDataFrame(rows, crs=CRS)

print(f"\n  Total zones built: {len(eco_zones)}")
print(eco_zones[["zone_id","zone_name","area_km2"]].to_string(index=False))


# ── EXPORT AS GEOJSON ─────────────────────────────────────────────────────

eco_zones.to_file(OUTPUT_FILE, driver="GeoJSON")
print(f"\n✅ GeoJSON saved → {OUTPUT_FILE}")

# ── EXPORT AS SHAPEFILE ───────────────────────────────────────────────────
# Shapefile column names are limited to 10 characters — rename long columns

shp_dir  = ZONE_DIR / "ecological_zones_5class"
shp_dir.mkdir(exist_ok=True)
shp_file = shp_dir / "ecological_zones_5class.shp"

shp_gdf = eco_zones.rename(columns={
    "zone_name_fr"  : "zone_fr",
    "source_zone"   : "src_zone",
    "rainfall_mm_yr": "rainfall",
})

shp_gdf.to_file(shp_file, driver="ESRI Shapefile")
print(f"✅ Shapefile saved → {shp_file}")

print(f"\n   Features : {len(eco_zones)}")
print(f"   CRS      : {eco_zones.crs}")
print(f"   Columns  : {list(eco_zones.columns)}")

print("""
══════════════════════════════════════════════════════════════
OUTPUT FILES:
  ecological_zones_5class.geojson        <- upload to GEE
  ecological_zones_5class/
    ecological_zones_5class.shp          <- open in QGIS
    ecological_zones_5class.dbf
    ecological_zones_5class.prj
    ecological_zones_5class.shx

NEXT STEP — Upload GeoJSON to GEE as asset:
  1. Go to https://code.earthengine.google.com/
  2. Assets tab -> New -> Table upload
  3. Select: ecological_zones_5class.geojson
  4. Asset ID: projects/ee-desmond/assets/ecological_zones_5class
  5. Click Upload

Once ingested your JS dashboard will work with:
  ECO_ZONES_READY = true
══════════════════════════════════════════════════════════════
""")

Loading shapefiles …
  ✓  Sahelian-Desert  (1 feature(s) dissolved)
  ✓  Soudanian_dissolved  (1 feature(s) dissolved)
  ✓  Guinea_dissolved  (1 feature(s) dissolved)

Building ecological zones …
  ✓  Saharian              area = 365,576 km²
  ✓  Sahelian              area = 1,049,125 km²
  ✓  Soudanian             area = 1,467,949 km²
  ✓  Guinean               area = 1,588,085 km²
  ✓  Guineo-Congolean      area = 829,254 km²

  Total zones built: 5
 zone_id        zone_name  area_km2
       1         Saharian  365576.0
       2         Sahelian 1049125.5
       3        Soudanian 1467949.3
       4          Guinean 1588085.4
       5 Guineo-Congolean  829253.9

✅ GeoJSON saved → C:\Users\Gebruiker\OneDrive\Spain\Paper 1\Cooling-Degree-Day-in-West-Africa\Maps\Climate Zone\ecological_zones_5class.geojson
✅ Shapefile saved → C:\Users\Gebruiker\OneDrive\Spain\Paper 1\Cooling-Degree-Day-in-West-Africa\Maps\Climate Zone\ecological_zones_5class\ecological_zones_5class.shp

   Features : 5
